In [1]:
import numpy as np
import torch

In [2]:
def create_W2(beta, J, h):
    # 計算 cosh 和 sinh 部分
    cosh = np.cosh(beta * J)
    sinh = np.sinh(beta * J)
    
    # 建立矩陣
    W_s = torch.tensor([
        [np.sqrt(cosh) * np.exp(beta * h/2), np.sqrt(sinh) * np.exp(beta * h/2)],
        [np.sqrt(cosh) * np.exp(-beta * h/2), -np.sqrt(sinh) * np.exp(-beta * h/2)]
    ])
    
    return W_s

def create_W(beta, J, h):
    # 計算 cosh 和 sinh 部分
    cosh_plus = np.cosh(beta * J + beta * h)
    sinh_plus = np.sinh(beta * J + beta * h)
    cosh_minus = np.cosh(beta * J - beta * h)
    sinh_minus = np.sinh(beta * J - beta * h)
    
    # 建立矩陣
    W_s = torch.tensor([
        [np.sqrt(cosh_plus), np.sqrt(sinh_plus)],
        [np.sqrt(cosh_minus), -np.sqrt(sinh_minus)]
    ])
    
    return W_s


temp = 4
beta = 1/temp
J = 1.0     # 自旋耦合
h = 0.1     # 外場

W_s = create_W(beta, J, h)
print(W_s)

W_s = create_W2(beta, J, h)
print(W_s)



tensor([[ 1.0188,  0.5277],
        [ 1.0126, -0.4763]], dtype=torch.float64)
tensor([[ 1.0284,  0.5089],
        [ 1.0030, -0.4964]], dtype=torch.float64)


In [3]:
temp = 4
beta = 1/temp
J = 1.0     # 自旋耦合
h = 0.1     # 外場

def get_M(beta, J = 1):
    # 計算 cosh 和 sinh 部分
    cosh = np.cosh(beta * J)
    sinh = np.sinh(beta * J)
    
    # 建立矩陣
    W_s = torch.tensor([
        [np.sqrt(cosh), np.sqrt(sinh)],
        [np.sqrt(cosh), -np.sqrt(sinh)]
    ])
    
    return W_s

M = get_M(beta)
print(M)

tensor([[ 1.0156,  0.5026],
        [ 1.0156, -0.5026]], dtype=torch.float64)


In [4]:
def get_W(beta, J=1, h=0):
    """
    Calculate the matrix W based on beta, J, and h.
    
    Parameters:
        beta (float): Inverse temperature.
        J (float): Interaction strength (default is 1).
        h (float): External field strength (default is 0).

    Returns:
        torch.Tensor: Matrix W.
    """
    sq_cosh = np.sqrt(np.cosh(beta * J))
    sq_sinh = np.sqrt(np.sinh(beta * J))
    W = torch.tensor([
        [sq_cosh, sq_sinh],
        [sq_cosh, -sq_sinh]
    ])
    return W


#可能Wh是正確的包含h的作法
def get_Wh(beta, J, h):
    """
    Calculate the matrix Wh incorporating the external field h.
    
    Parameters:
        beta (float): Inverse temperature.
        J (float): Interaction strength.
        h (float): External field strength.

    Returns:
        torch.Tensor: Matrix Wh.
    """
    cosh = np.cosh(beta * J)
    sinh = np.sinh(beta * J)

    Wh = torch.tensor([
        [np.sqrt(cosh) * np.exp(beta * h / 2), np.sqrt(sinh) * np.exp(beta * h / 2)],
        [np.sqrt(cosh) * np.exp(-beta * h / 2), -np.sqrt(sinh) * np.exp(-beta * h / 2)]
    ])
    return Wh


# 上左右下
def get_T_bare(beta, J=1, h=0, get_W=get_W):
    """
    Calculate the bare transfer matrix T_bare.
    
    Parameters:
        beta (float): Inverse temperature.
        J (float): Interaction strength (default is 1).
        h (float): External field strength (default is 0).
        get_W (callable): Function to calculate W or Wh (default is get_W).

    Returns:
        torch.Tensor: Bare transfer matrix T_bare.
    """
    W = get_W(beta, J=J, h=h)
    T_bare = torch.einsum('ai,aj,ak,al->ijkl', W, W, W, W)
    return T_bare


In [5]:
print(get_T_bare(1/4))
print(get_T_bare(1/3))
print(get_T_bare(1/2))

print(get_T_bare(1/4, h=-0.1, get_W = get_Wh))

tensor([[[[2.1276, 0.0000],
          [0.0000, 0.5211]],

         [[0.0000, 0.5211],
          [0.5211, 0.0000]]],


        [[[0.0000, 0.5211],
          [0.5211, 0.0000]],

         [[0.5211, 0.0000],
          [0.0000, 0.1276]]]], dtype=torch.float64)
tensor([[[[2.2306, 0.0000],
          [0.0000, 0.7172]],

         [[0.0000, 0.7172],
          [0.7172, 0.0000]]],


        [[[0.0000, 0.7172],
          [0.7172, 0.0000]],

         [[0.7172, 0.0000],
          [0.0000, 0.2306]]]], dtype=torch.float64)
tensor([[[[2.5431, 0.0000],
          [0.0000, 1.1752]],

         [[0.0000, 1.1752],
          [1.1752, 0.0000]]],


        [[[0.0000, 1.1752],
          [1.1752, 0.0000]],

         [[1.1752, 0.0000],
          [0.0000, 0.5431]]]], dtype=torch.float64)
tensor([[[[ 2.1303, -0.0527],
          [-0.0527,  0.5217]],

         [[-0.0527,  0.5217],
          [ 0.5217, -0.0129]]],


        [[[-0.0527,  0.5217],
          [ 0.5217, -0.0129]],

         [[ 0.5217, -0.0129],
          [-0.

In [6]:
def merge_y(Tup, Tdn, combine=False, trace=False):
    if trace:
        Tmerge = torch.einsum('abcd,defa->becf', Tup, Tdn)
        
        if combine:
            shape = Tmerge.shape
            Tmerge = Tmerge.contiguous().view(shape[0] * shape[1], shape[2] * shape[3])
    else:
        Tmerge = torch.einsum('abcd,defg->abecfg', Tup, Tdn)
        
        if combine:
            shape = Tmerge.shape
            Tmerge = Tmerge.contiguous().view(shape[0], shape[1] * shape[2], shape[3] * shape[4], shape[5])
            
    return Tmerge

In [7]:
MaxL = 4
Temp = np.linspace(2.26,4,100)
W = np.ones((MaxL+1,len(Temp),2))
E = np.ones((MaxL+1,len(Temp),2))
Tc = 2/np.log(1+np.sqrt(2))

In [8]:
TL = get_T_bare(4)
TL_Trace = merge_y(TL,TL,True,True)

In [9]:
%%time

for i,temp in enumerate(Temp):
    T_bare = get_T_bare(1/temp)
    TL = T_bare
    for j in range(2,MaxL+1):
        TL_Trace = merge_y(TL,TL,True,True)
        eigvals, eigvecs = torch.linalg.eigh(TL_Trace)
        W[j,i,:] = eigvals[-2:]
        del TL_Trace
        if j != MaxL:
            TL = merge_y(TL,TL,True)
E = -np.log(W)
Corr_len = 1/(E[:,:,-2]-E[:,:,-1])

CPU times: user 2.28 s, sys: 1.76 s, total: 4.04 s
Wall time: 686 ms


<timed exec>:12: RuntimeWarning: divide by zero encountered in divide


In [10]:
Corr_len

array([[        inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,         inf,         inf,
                inf,         inf,         inf,     

In [11]:
def merge_y_truncate(Tup, Tdn, dcut, niter = 10):
    if ((Tup.shape[1] * Tdn.shape[1]) < dcut):
        return merge_y(Tup, Tdn, True)
    Tmerge_pure = torch.einsum('abcd,defg,achi,iejg->bfhj', Tup, Tdn, Tup, Tdn)
    Tmerge_pure_shape = Tmerge_pure.shape
    Tmerge_pure = Tmerge_pure.contiguous().view(
        Tmerge_pure_shape[0] * Tmerge_pure_shape[1],Tmerge_pure_shape[2] * Tmerge_pure_shape[3]
    )
    U, S, V = torch.svd_lowrank(Tmerge_pure, q=dcut, niter=niter)
    U = U.view(Tmerge_pure_shape[0], Tmerge_pure_shape[1], dcut)
    Tmerge = torch.einsum('abcd,defg,bfh,cei->ahig', Tup, Tdn, U, U)
    return Tmerge

In [12]:
temp = 2
dcut = 8

TL = get_T_bare(1/temp)
print(TL)
TL = merge_y_truncate(TL,TL,dcut)
print(TL)
TL = merge_y_truncate(TL,TL,dcut)
print(TL)
TL = merge_y_truncate(TL,TL,dcut)
print(TL)
TL = merge_y_truncate(TL,TL,dcut)
print(TL)

tensor([[[[2.5431, 0.0000],
          [0.0000, 1.1752]],

         [[0.0000, 1.1752],
          [1.1752, 0.0000]]],


        [[[0.0000, 1.1752],
          [1.1752, 0.0000]],

         [[1.1752, 0.0000],
          [0.0000, 0.5431]]]], dtype=torch.float64)
tensor([[[[6.4673, 0.0000],
          [0.0000, 2.9886],
          [0.0000, 1.3811],
          [1.3811, 0.0000]],

         [[0.0000, 2.9886],
          [2.9886, 0.0000],
          [1.3811, 0.0000],
          [0.0000, 0.6382]],

         [[0.0000, 1.3811],
          [1.3811, 0.0000],
          [2.9886, 0.0000],
          [0.0000, 1.3811]],

         [[1.3811, 0.0000],
          [0.0000, 0.6382],
          [0.0000, 1.3811],
          [1.3811, 0.0000]]],


        [[[0.0000, 1.3811],
          [1.3811, 0.0000],
          [2.9886, 0.0000],
          [0.0000, 1.3811]],

         [[1.3811, 0.0000],
          [0.0000, 0.6382],
          [0.0000, 1.3811],
          [1.3811, 0.0000]],

         [[2.9886, 0.0000],
          [0.0000, 1.3811],
  

In [13]:
def merge_x(TL, TR, combine=False, trace=False):
    if trace:
        Tmerge = torch.einsum('abca,ecfe->bf', TL, TR)
    else:
        Tmerge = torch.einsum('abcd,ecfg->eabfgd', TL, TR)
        
        if combine:
            shape = Tmerge.shape
            Tmerge = Tmerge.contiguous().view(shape[0] * shape[1], shape[2], shape[3], shape[4] * shape[5])
            
    return Tmerge

def merge_x_truncate(TL, TR, dcut, niter = 10):
    if ((TL.shape[0] * TR.shape[0]) < dcut):
        return merge_x(TL, TR, True)
    Tmerge_pure = torch.einsum('abcd,ecfg,hbid,jifg->aehj', TL, TR, TL, TR)
    Tmerge_pure_shape = Tmerge_pure.shape
    Tmerge_pure = Tmerge_pure.contiguous().view(
        Tmerge_pure_shape[0] * Tmerge_pure_shape[1],Tmerge_pure_shape[2] * Tmerge_pure_shape[3]
    )
    U, S, V = torch.svd_lowrank(Tmerge_pure, q=dcut, niter=niter)
    U = U.view(Tmerge_pure_shape[0], Tmerge_pure_shape[1], dcut)
    Tmerge = torch.einsum('abcd,ecfg,aeh,dgi->hbfi', TL, TR, U, U)
    return Tmerge

In [14]:
temp = 2
dcut = 2

TL = get_T_bare(1/temp)
print(TL)
TL = merge_x_truncate(TL,TL,dcut)
print(TL)
TL = merge_x_truncate(TL,TL,dcut)
print(TL)
TL = merge_x_truncate(TL,TL,dcut)
print(TL)
TL = merge_x_truncate(TL,TL,dcut)
print(TL)

tensor([[[[2.5431, 0.0000],
          [0.0000, 1.1752]],

         [[0.0000, 1.1752],
          [1.1752, 0.0000]]],


        [[[0.0000, 1.1752],
          [1.1752, 0.0000]],

         [[1.1752, 0.0000],
          [0.0000, 0.5431]]]], dtype=torch.float64)
tensor([[[[ 6.7980e+00, -4.4055e-19],
          [-1.6488e-19,  3.3772e+00]],

         [[-9.4030e-20,  3.3772e+00],
          [ 2.0788e+00, -1.3273e-19]]],


        [[[-4.4055e-19,  4.3697e+00],
          [ 3.3772e+00, -3.3500e-19]],

         [[ 3.3772e+00, -3.3500e-19],
          [-1.3273e-19,  2.0193e+00]]]], dtype=torch.float64)
tensor([[[[ 5.0174e+01, -1.4809e-16],
          [ 1.3685e-14,  2.5600e+01]],

         [[ 1.2436e-14,  2.5600e+01],
          [ 1.2840e+01,  1.5955e-15]]],


        [[[-1.4809e-16,  4.1111e+01],
          [ 2.5600e+01, -1.0312e-14]],

         [[ 2.5600e+01, -1.0311e-14],
          [ 1.5955e-15,  1.5603e+01]]]], dtype=torch.float64)
tensor([[[[ 2.8726e+03,  3.2022e-11],
          [-1.1500e-10,  1.5136e+0

In [15]:
TL = get_T_bare(1/temp)
print(TL)
TL = merge_x(TL,TL,True,False)
print(TL)
print(TL.shape)

tensor([[[[2.5431, 0.0000],
          [0.0000, 1.1752]],

         [[0.0000, 1.1752],
          [1.1752, 0.0000]]],


        [[[0.0000, 1.1752],
          [1.1752, 0.0000]],

         [[1.1752, 0.0000],
          [0.0000, 0.5431]]]], dtype=torch.float64)
tensor([[[[6.4673, 0.0000, 0.0000, 1.3811],
          [0.0000, 1.3811, 2.9886, 0.0000]],

         [[0.0000, 2.9886, 1.3811, 0.0000],
          [1.3811, 0.0000, 0.0000, 1.3811]]],


        [[[0.0000, 2.9886, 1.3811, 0.0000],
          [1.3811, 0.0000, 0.0000, 1.3811]],

         [[2.9886, 0.0000, 0.0000, 0.6382],
          [0.0000, 0.6382, 1.3811, 0.0000]]],


        [[[0.0000, 1.3811, 2.9886, 0.0000],
          [2.9886, 0.0000, 0.0000, 0.6382]],

         [[1.3811, 0.0000, 0.0000, 1.3811],
          [0.0000, 1.3811, 0.6382, 0.0000]]],


        [[[1.3811, 0.0000, 0.0000, 1.3811],
          [0.0000, 1.3811, 0.6382, 0.0000]],

         [[0.0000, 0.6382, 1.3811, 0.0000],
          [1.3811, 0.0000, 0.0000, 0.2949]]]], dtype=torch.float

In [16]:
%%time
dcut = 8

for i,temp in enumerate(Temp):
    T_bare = get_T_bare(1/temp)
    TL = T_bare
    for j in range(2,MaxL+1):
        #print(j)
        TLx = merge_x_truncate(TL,TL,dcut)
        TL_Trace = merge_y(TLx,TLx,True,True)
        print(TL_Trace.shape)
        eigvals, eigvecs = torch.linalg.eigh(TL_Trace)
        W[j,i,:] = eigvals[-2:]
        TL = merge_y_truncate(TLx,TLx,dcut)
        TL = TL/torch.mean(torch.abs(TL))
E = -np.log(W)
Corr_len_2 = 1/(E[:,:,-2]-E[:,:,-1])

torch.Size([4, 4])
torch.Size([16, 16])
torch.Size([64, 64])
torch.Size([4, 4])
torch.Size([16, 16])
torch.Size([64, 64])
torch.Size([4, 4])
torch.Size([16, 16])
torch.Size([64, 64])
torch.Size([4, 4])
torch.Size([16, 16])
torch.Size([64, 64])
torch.Size([4, 4])
torch.Size([16, 16])
torch.Size([64, 64])
torch.Size([4, 4])
torch.Size([16, 16])
torch.Size([64, 64])
torch.Size([4, 4])
torch.Size([16, 16])
torch.Size([64, 64])
torch.Size([4, 4])
torch.Size([16, 16])
torch.Size([64, 64])
torch.Size([4, 4])
torch.Size([16, 16])
torch.Size([64, 64])
torch.Size([4, 4])
torch.Size([16, 16])
torch.Size([64, 64])
torch.Size([4, 4])
torch.Size([16, 16])
torch.Size([64, 64])
torch.Size([4, 4])
torch.Size([16, 16])
torch.Size([64, 64])
torch.Size([4, 4])
torch.Size([16, 16])
torch.Size([64, 64])
torch.Size([4, 4])
torch.Size([16, 16])
torch.Size([64, 64])
torch.Size([4, 4])
torch.Size([16, 16])
torch.Size([64, 64])
torch.Size([4, 4])
torch.Size([16, 16])
torch.Size([64, 64])
torch.Size([4, 4])
torch

<timed exec>:16: RuntimeWarning: divide by zero encountered in divide


In [17]:
%%time

for i,temp in enumerate(Temp):
    T_bare = get_T_bare(1/temp)
    TL = T_bare
    for j in range(2,MaxL+1):
        TL_Trace = merge_y(TL,TL,True,True)
        eigvals, eigvecs = torch.linalg.eigh(TL_Trace)
        W[j,i,:] = eigvals[-2:]
        del TL_Trace
        if j != MaxL:
            TL = merge_y(TL,TL,True)
E = -np.log(W)
Corr_len = 1/(E[:,:,-2]-E[:,:,-1])

CPU times: user 2.27 s, sys: 1.8 s, total: 4.08 s
Wall time: 716 ms


<timed exec>:12: RuntimeWarning: divide by zero encountered in divide
